# 📖 Notebook 3: Rate Limiting Basics

## Why Rate Limiting?

Imagine you run a popular restaurant. If you let **unlimited people** in at once, the kitchen gets
overwhelmed, food takes forever, and everyone has a bad experience. Rate limiting is like having a
**host at the door** who controls how many people come in per hour — it keeps things running smoothly
for everyone.

The same idea applies to APIs. Without rate limiting, a single client could send **millions of requests**
and crash your server, leaving every other user without service.

## 🎯 Learning Objectives

By the end of this notebook, you will:

1. **Understand** why rate limiting is essential for any public API
2. **Learn** how the fixed-window algorithm works (the simplest rate limiting approach)
3. **Read** rate-limit headers (`X-RateLimit-Limit`, `X-RateLimit-Remaining`, `X-RateLimit-Reset`)
4. **Handle** `429 Too Many Requests` responses gracefully as a client
5. **Compare** different rate limiting algorithms (fixed window, sliding window, token bucket, leaky bucket)

## ⚙️ Setup

Before running this notebook, make sure the lab environment is running:

```bash
# From the api-design lab directory
docker-compose up -d
```

**Kernel Selection (VS Code):**
1. Click the kernel picker in the top-right corner of this notebook
2. Select the `.venv` Python environment from this lab folder
3. If the kernel doesn't appear, reload the VS Code window (`Cmd+Shift+P` → "Reload Window")

**Services used in this notebook:**

| Service | URL |
|---------|-----|
| FastAPI server | http://localhost:8000 |
| API docs (Swagger) | http://localhost:8000/docs |
| Rate-limited endpoint | GET /limited/events |
| Reset endpoint | POST /limited/reset |

In [1]:
# === Connection Setup ===
# We only need 3 standard Python libraries — no extra installs!

import requests  # For making HTTP requests to our API
import time      # For measuring time and adding delays
import json      # For pretty-printing JSON responses

# The base URL of our FastAPI server running in Docker
BASE_URL = "http://localhost:8000"

# Step 1: Reset any existing rate limits so we start fresh
# This endpoint clears all rate-limit counters on the server
reset_response = requests.post(f"{BASE_URL}/limited/reset")
print(f"Reset status: {reset_response.status_code}")

# Step 2: Test that the server is reachable
test_response = requests.get(f"{BASE_URL}/limited/events")
print(f"Test request status: {test_response.status_code}")
print(f"\n✅ Connected to API server at {BASE_URL}")

Reset status: 200
Test request status: 200

✅ Connected to API server at http://localhost:8000


## 🤔 Why Rate Limiting?

### Without Rate Limiting

Without rate limiting, **anyone** can send as many requests as they want:

- A **buggy client** with an infinite loop could send thousands of requests per second
- A **web scraper** could hammer your API trying to download all your data
- A **DDoS attack** (Distributed Denial of Service) could flood your server with traffic
- One greedy user could use up all the server's resources, leaving nothing for others

### The Bouncer Analogy 🚪

Think of rate limiting like a **bouncer at a club**:

- The club has a rule: **max 10 people can enter per hour**
- The bouncer keeps count of how many people entered in the current hour
- If someone tries to enter and the count is already at 10, they're told: *"Sorry, try again later"*
- When the next hour starts, the counter resets to 0

That's exactly how a **fixed-window rate limiter** works!

### Real-World Examples

| API | Rate Limit | Window |
|-----|-----------|--------|
| Twitter/X API | 300 requests | 15 minutes |
| GitHub API | 5,000 requests | 1 hour |
| Google Maps API | 50 requests | 1 second |
| Stripe API | 100 requests | 1 second |

### What Rate Limiting Protects Against

1. **DDoS attacks** — Limits damage from traffic floods
2. **Buggy clients** — Prevents accidental infinite loops from crashing your server
3. **Web scrapers** — Slows down unauthorized data collection
4. **Fair access** — Ensures every user gets a fair share of server resources

## 🔧 How Rate Limiting Works

Our API uses the **fixed-window** algorithm, the simplest form of rate limiting:

```
Time:  0s -------------- 60s -------------- 120s
       |   Window 1      |    Window 2      |
       | Count: 0→1→2... |  Count resets→0  |
       | Limit: 10       |  Limit: 10       |
```

**How it works step by step:**

1. Time is divided into **windows** (e.g., 60-second windows)
2. Each request **increments a counter** for the current window
3. If the counter exceeds the limit → return **429 Too Many Requests**
4. When the window expires → the counter **resets to 0**

**The server tells you your status via HTTP headers:**

| Header | Meaning | Example |
|--------|---------|----------|
| `X-RateLimit-Limit` | Max requests allowed per window | `10` |
| `X-RateLimit-Remaining` | Requests you have left | `7` |
| `X-RateLimit-Reset` | Unix timestamp when the window resets | `1700000060` |

Let's see this in action! 👇

In [2]:
# === Seeing Rate Limits in Action ===
# Let's make a single request and inspect the rate-limit headers

# First, reset the limits so we start with a clean slate
requests.post(f"{BASE_URL}/limited/reset")

# Make one request to the rate-limited endpoint
response = requests.get(f"{BASE_URL}/limited/events")

# Print the HTTP status code (should be 200 = success)
print(f"Status Code: {response.status_code}")
print()

# Print the rate-limit headers — these tell us our current status
print("=== Rate Limit Headers ===")

# X-RateLimit-Limit: The maximum number of requests allowed per window
limit = response.headers.get("X-RateLimit-Limit", "N/A")
print(f"X-RateLimit-Limit:     {limit}  ← Max requests per window")

# X-RateLimit-Remaining: How many requests we have left before hitting the limit
remaining = response.headers.get("X-RateLimit-Remaining", "N/A")
print(f"X-RateLimit-Remaining: {remaining}  ← Requests remaining")

# X-RateLimit-Reset: Unix timestamp when the rate limit window resets
reset_ts = response.headers.get("X-RateLimit-Reset", "N/A")
print(f"X-RateLimit-Reset:     {reset_ts}  ← Window reset time (unix timestamp)")

print()
print("💡 After 1 request, we used 1 of our 10 allowed requests.")
print(f"   We have {remaining} requests remaining in this window.")

Status Code: 200

=== Rate Limit Headers ===
X-RateLimit-Limit:     10  ← Max requests per window
X-RateLimit-Remaining: 9  ← Requests remaining
X-RateLimit-Reset:     1776514500  ← Window reset time (unix timestamp)

💡 After 1 request, we used 1 of our 10 allowed requests.
   We have 9 requests remaining in this window.


In [3]:
# === Watching the Counter Decrease ===
# Let's make 5 requests and watch X-RateLimit-Remaining go down

# Reset limits first
requests.post(f"{BASE_URL}/limited/reset")

print("Making 5 requests and watching the remaining count decrease...")
print()

for i in range(1, 6):
    # Make a request to the rate-limited endpoint
    response = requests.get(f"{BASE_URL}/limited/events")
    
    # Read the remaining count from the header
    remaining = response.headers.get("X-RateLimit-Remaining", "?")
    
    # Print a nice visual showing the count going down
    bar = "█" * int(remaining) + "░" * (10 - int(remaining))
    print(f"  Request {i}: Status {response.status_code} | Remaining: {remaining}/10 | {bar}")

print()
print("📉 See how the remaining count decreases with each request!")
print("   Each request 'uses up' one of our allowed requests for this window.")

Making 5 requests and watching the remaining count decrease...

  Request 1: Status 200 | Remaining: 9/10 | █████████░
  Request 2: Status 200 | Remaining: 8/10 | ████████░░
  Request 3: Status 200 | Remaining: 7/10 | ███████░░░
  Request 4: Status 200 | Remaining: 6/10 | ██████░░░░
  Request 5: Status 200 | Remaining: 5/10 | █████░░░░░

📉 See how the remaining count decreases with each request!
   Each request 'uses up' one of our allowed requests for this window.


In [4]:
# === Hitting the Limit ===
# Our limit is 10 requests per 60 seconds.
# Let's try to make 12 requests and see what happens!

# Reset limits for a clean start
requests.post(f"{BASE_URL}/limited/reset")

# Track how many succeed vs get rate-limited
succeeded = 0
rate_limited = 0
last_rejected_body = None

print("Making 12 requests (limit is 10)...")
print()

for i in range(1, 13):
    response = requests.get(f"{BASE_URL}/limited/events")
    
    if response.status_code == 200:
        # Request succeeded — we're still under the limit
        remaining = response.headers.get("X-RateLimit-Remaining", "?")
        print(f"  Request {i:2d}: ✅ 200 OK        | Remaining: {remaining}")
        succeeded += 1
    elif response.status_code == 429:
        # Rate limited! The server is saying "slow down!"
        print(f"  Request {i:2d}: ❌ 429 Too Many Requests")
        rate_limited += 1
        last_rejected_body = response.json()
    else:
        print(f"  Request {i:2d}: ⚠️  {response.status_code} (unexpected)")

print()
print("=" * 50)
print(f"📊 Summary: {succeeded} requests succeeded, {rate_limited} were rate limited")

# Show what the 429 response body looks like
if last_rejected_body:
    print()
    print("The 429 response body looks like this:")
    print(json.dumps(last_rejected_body, indent=2))

Making 12 requests (limit is 10)...

  Request  1: ✅ 200 OK        | Remaining: 9
  Request  2: ✅ 200 OK        | Remaining: 8
  Request  3: ✅ 200 OK        | Remaining: 7
  Request  4: ✅ 200 OK        | Remaining: 6
  Request  5: ✅ 200 OK        | Remaining: 5
  Request  6: ✅ 200 OK        | Remaining: 4
  Request  7: ✅ 200 OK        | Remaining: 3
  Request  8: ✅ 200 OK        | Remaining: 2
  Request  9: ✅ 200 OK        | Remaining: 1
  Request 10: ✅ 200 OK        | Remaining: 0
  Request 11: ❌ 429 Too Many Requests
  Request 12: ❌ 429 Too Many Requests

📊 Summary: 10 requests succeeded, 2 were rate limited

The 429 response body looks like this:
{
  "detail": "Too Many Requests"
}


## 🧮 The Fixed-Window Algorithm

Let's understand the fixed-window algorithm by **building one ourselves** in pure Python.

The idea is simple:

1. **Divide time into fixed windows** (e.g., every 60 seconds is a new window)
2. **Count requests** in the current window
3. **If count > limit** → reject the request
4. **When the window expires** → reset the counter

```
Window 1 (0s-60s)        Window 2 (60s-120s)
┌────────────────────┐   ┌────────────────────┐
│ Req1 Req2 ... Req10│   │ Req1 Req2 ...      │
│ count = 10 (FULL!) │   │ count resets to 0   │
│ Req11 → REJECTED   │   │ accepts again!      │
└────────────────────┘   └────────────────────┘
```

### ⚠️ The Edge Case: Burst at Window Boundary

Fixed-window has a well-known problem. Imagine the window resets every 60 seconds:

- At second **59**: a client sends **10 requests** (uses up the entire window)
- At second **61**: the window resets, so the client sends **10 more requests**
- Result: **20 requests in just 2 seconds!** Even though the limit is 10 per minute.

This is called the **burst at window boundary** problem. We'll see this in code below.

In [5]:
# === Building a Fixed-Window Rate Limiter in Pure Python ===
# This doesn't use the server — it's a standalone implementation to learn the algorithm.

class FixedWindowRateLimiter:
    """
    A simple fixed-window rate limiter.
    
    It divides time into windows (e.g., 60 seconds each) and counts
    how many requests happen in the current window. If the count exceeds
    the limit, new requests are rejected until the window resets.
    """
    
    def __init__(self, max_requests, window_seconds):
        self.max_requests = max_requests      # e.g., 10 requests
        self.window_seconds = window_seconds    # e.g., 60 seconds
        self.request_count = 0                  # How many requests in current window
        self.window_start = time.time()         # When the current window started
    
    def allow_request(self):
        """Check if a request should be allowed or rejected."""
        now = time.time()
        
        # Step 1: Check if the current window has expired
        if now - self.window_start >= self.window_seconds:
            # Window expired! Reset the counter and start a new window
            self.request_count = 0
            self.window_start = now
        
        # Step 2: Check if we're under the limit
        if self.request_count < self.max_requests:
            # Under the limit — allow the request
            self.request_count += 1
            return True   # ✅ Request allowed
        else:
            # Over the limit — reject the request
            return False  # ❌ Request rejected (429)
    
    def get_remaining(self):
        """How many requests are left in the current window?"""
        return max(0, self.max_requests - self.request_count)


# Test our rate limiter: allow 5 requests per window
limiter = FixedWindowRateLimiter(max_requests=5, window_seconds=60)

print("Testing our fixed-window rate limiter (limit: 5 requests)")
print()

for i in range(1, 9):
    allowed = limiter.allow_request()
    remaining = limiter.get_remaining()
    status = "✅ Allowed" if allowed else "❌ Rejected (429)"
    print(f"  Request {i}: {status} | Remaining: {remaining}")

print()
print("💡 Requests 1-5 were allowed. Requests 6-8 were rejected.")
print("   The counter would reset when the 60-second window expires.")

Testing our fixed-window rate limiter (limit: 5 requests)

  Request 1: ✅ Allowed | Remaining: 4
  Request 2: ✅ Allowed | Remaining: 3
  Request 3: ✅ Allowed | Remaining: 2
  Request 4: ✅ Allowed | Remaining: 1
  Request 5: ✅ Allowed | Remaining: 0
  Request 6: ❌ Rejected (429) | Remaining: 0
  Request 7: ❌ Rejected (429) | Remaining: 0
  Request 8: ❌ Rejected (429) | Remaining: 0

💡 Requests 1-5 were allowed. Requests 6-8 were rejected.
   The counter would reset when the 60-second window expires.


In [6]:
# === Demonstrating the Burst at Window Boundary Problem ===
# This shows why fixed-window isn't perfect.

class SimulatedFixedWindowLimiter:
    """
    Same as above, but accepts a simulated 'current time' so we can
    fast-forward through time for demonstration purposes.
    """
    
    def __init__(self, max_requests, window_seconds):
        self.max_requests = max_requests
        self.window_seconds = window_seconds
        self.request_count = 0
        self.window_start = 0  # Start at simulated time 0
    
    def allow_request(self, current_time):
        # Check if window expired
        if current_time - self.window_start >= self.window_seconds:
            self.request_count = 0
            self.window_start = (current_time // self.window_seconds) * self.window_seconds
        
        if self.request_count < self.max_requests:
            self.request_count += 1
            return True
        return False


# Create a limiter: 10 requests per 60-second window
limiter = SimulatedFixedWindowLimiter(max_requests=10, window_seconds=60)

print("=== Burst at Window Boundary Demo ===")
print("Limit: 10 requests per 60-second window")
print()

# Scenario: send 10 requests at second 59 (end of window 1)
print("--- At second 59 (end of Window 1) ---")
allowed_count = 0
for i in range(10):
    if limiter.allow_request(current_time=59):
        allowed_count += 1
print(f"  Sent 10 requests → {allowed_count} allowed ✅")

# Now send 10 more at second 61 (start of window 2 — counter resets!)
print()
print("--- At second 61 (start of Window 2, counter resets!) ---")
allowed_count = 0
for i in range(10):
    if limiter.allow_request(current_time=61):
        allowed_count += 1
print(f"  Sent 10 requests → {allowed_count} allowed ✅")

print()
print("⚠️  Result: 20 requests allowed in just 2 seconds!")
print("   Even though the limit is 10 per 60 seconds.")
print("   This is the 'burst at window boundary' problem.")
print("   Sliding window algorithms solve this issue.")

=== Burst at Window Boundary Demo ===
Limit: 10 requests per 60-second window

--- At second 59 (end of Window 1) ---
  Sent 10 requests → 10 allowed ✅

--- At second 61 (start of Window 2, counter resets!) ---
  Sent 10 requests → 10 allowed ✅

⚠️  Result: 20 requests allowed in just 2 seconds!
   Even though the limit is 10 per 60 seconds.
   This is the 'burst at window boundary' problem.
   Sliding window algorithms solve this issue.


## 🛡️ Handling Rate Limits as a Client

As a client (someone calling an API), you should **always** handle rate limits gracefully.

**Best practices:**

1. **Check the headers** — Before making requests, look at `X-RateLimit-Remaining`
2. **Don't ignore 429 responses** — They're telling you to slow down
3. **Use exponential backoff** — Wait longer between each retry:
   - 1st retry: wait 1 second
   - 2nd retry: wait 2 seconds
   - 3rd retry: wait 4 seconds
4. **Respect the `Reset` header** — It tells you exactly when you can try again

Let's build a **smart request function** that handles rate limits automatically!

In [7]:
# === Smart Request Function with Exponential Backoff ===
# This function automatically handles rate limits (429 responses)

def smart_request(url, max_retries=3):
    """
    Makes a GET request to the given URL with automatic rate-limit handling.
    
    If we get a 429 (rate limited), it waits and retries using exponential backoff:
      - 1st retry: wait 1 second
      - 2nd retry: wait 2 seconds
      - 3rd retry: wait 4 seconds
    
    Returns the response (or None if all retries fail).
    """
    for attempt in range(max_retries + 1):
        response = requests.get(url)
        
        if response.status_code == 200:
            # Success! Return the response
            return response
        
        elif response.status_code == 429:
            # Rate limited — we need to wait and retry
            if attempt < max_retries:
                # Exponential backoff: 1s, 2s, 4s, 8s...
                wait_time = 2 ** attempt
                print(f"    ⏳ Rate limited! Waiting {wait_time}s before retry {attempt + 1}/{max_retries}...")
                time.sleep(wait_time)
            else:
                print(f"    ❌ Rate limited and all {max_retries} retries exhausted.")
                return response
        else:
            # Some other error — return as-is
            return response
    
    return response


# Demo: make 15 requests using the smart_request function
# Our limit is 10, so requests 11-15 will trigger retries

# Reset limits for a clean start
requests.post(f"{BASE_URL}/limited/reset")

print("Making 15 requests with smart_request() (limit is 10)...")
print("Requests beyond the limit will automatically retry with backoff.")
print()

succeeded = 0
failed = 0

for i in range(1, 16):
    print(f"  Request {i:2d}: ", end="")
    response = smart_request(f"{BASE_URL}/limited/events")
    
    if response and response.status_code == 200:
        remaining = response.headers.get("X-RateLimit-Remaining", "?")
        print(f"✅ 200 OK | Remaining: {remaining}")
        succeeded += 1
    else:
        print(f"❌ Failed after retries")
        failed += 1

print()
print(f"📊 Results: {succeeded} succeeded, {failed} failed after retries")
print()
print("💡 The smart_request function automatically backs off when rate limited.")
print("   In a real app, the wait-and-retry would eventually succeed once")
print("   the rate limit window resets.")

Making 15 requests with smart_request() (limit is 10)...
Requests beyond the limit will automatically retry with backoff.

  Request  1: ✅ 200 OK | Remaining: 9
  Request  2: ✅ 200 OK | Remaining: 8
  Request  3: ✅ 200 OK | Remaining: 7
  Request  4: ✅ 200 OK | Remaining: 6
  Request  5: ✅ 200 OK | Remaining: 5
  Request  6: ✅ 200 OK | Remaining: 4
  Request  7: ✅ 200 OK | Remaining: 3
  Request  8: ✅ 200 OK | Remaining: 2
  Request  9: ✅ 200 OK | Remaining: 1
  Request 10: ✅ 200 OK | Remaining: 0
  Request 11:     ⏳ Rate limited! Waiting 1s before retry 1/3...


    ⏳ Rate limited! Waiting 2s before retry 2/3...


    ⏳ Rate limited! Waiting 4s before retry 3/3...


    ❌ Rate limited and all 3 retries exhausted.
❌ Failed after retries
  Request 12:     ⏳ Rate limited! Waiting 1s before retry 1/3...


    ⏳ Rate limited! Waiting 2s before retry 2/3...


...

    ⏳ Rate limited! Waiting 4s before retry 3/3...


    ❌ Rate limited and all 3 retries exhausted.
❌ Failed after retries
  Request 15:     ⏳ Rate limited! Waiting 1s before retry 1/3...


    ⏳ Rate limited! Waiting 2s before retry 2/3...


    ⏳ Rate limited! Waiting 4s before retry 3/3...


    ❌ Rate limited and all 3 retries exhausted.
❌ Failed after retries

📊 Results: 10 succeeded, 5 failed after retries

💡 The smart_request function automatically backs off when rate limited.
   In a real app, the wait-and-retry would eventually succeed once
   the rate limit window resets.


## ⏲️ The `Retry-After` Header (The Standard)

`X-RateLimit-*` headers are widely used, but they're **not** part of the HTTP
standard — every vendor prefixes them differently (GitHub uses `X-RateLimit-*`,
Twitter used `x-rate-limit-*`, etc.).

There **is** a standard header for "try again later": [`Retry-After`](https://developer.mozilla.org/en-US/docs/Web/HTTP/Headers/Retry-After) (from RFC 6585 / RFC 7231).

- Value is either **seconds** (like `Retry-After: 30`) or an HTTP date.
- It's what HTTP libraries, proxies, and browsers already know how to respect.
- Our server sends it **only on 429 responses** — the spec says to include it
  specifically when a request was rejected.

Prefer `Retry-After` in your clients when it's present.


In [8]:
# Trigger a 429 and read the standard Retry-After header
requests.post(f"{BASE_URL}/limited/reset")

# Burn through the 10 allowed requests
for _ in range(10):
    requests.get(f"{BASE_URL}/limited/events")

# The 11th request gets rejected — look at the headers.
rejected = requests.get(f"{BASE_URL}/limited/events")
print(f"Status: {rejected.status_code}")
print("Relevant headers on 429:")
for h in ("Retry-After", "X-RateLimit-Limit", "X-RateLimit-Remaining", "X-RateLimit-Reset"):
    print(f"  {h:<22} = {rejected.headers.get(h)}")

print()
print(f"💡 Retry-After says: wait {rejected.headers.get('Retry-After')} second(s) before retrying.")
print("   A smart client would do exactly that — no guessing, no exponential backoff needed.")


Status: 429
Relevant headers on 429:
  Retry-After            = 1
  X-RateLimit-Limit      = 10
  X-RateLimit-Remaining  = 0
  X-RateLimit-Reset      = 1776514537

💡 Retry-After says: wait 1 second(s) before retrying.
   A smart client would do exactly that — no guessing, no exponential backoff needed.


## 🔑 Per-User vs Per-IP Rate Limiting

Limiting by IP is the simplest approach, but it breaks in two common cases:

1. **Shared IPs** (offices, mobile carriers, NAT) — one abusive user gets
   everyone on the same IP blocked.
2. **Authenticated clients** with API keys — we can, and *should*, count
   against each key individually so that a paying customer isn't punished for
   another customer's traffic.

Our server checks for an `X-API-Key` header: when present, the limiter counts
per key; otherwise it falls back to the client IP. Let's see it in action.


In [9]:
# Two different API keys get their own quotas, even from the same IP.
requests.post(f"{BASE_URL}/limited/reset")

def hit(api_key):
    return requests.get(
        f"{BASE_URL}/limited/events",
        headers={"X-API-Key": api_key},
    )

# Burn through 10 requests for key 'alice'
for _ in range(10):
    hit("alice")
resp_alice = hit("alice")  # 11th for alice → rejected
resp_bob = hit("bob")      # 1st for bob    → allowed (fresh quota)

print(f"alice 11th request : {resp_alice.status_code}  (alice exhausted her quota)")
print(f"bob   1st  request : {resp_bob.status_code}  (bob has his own bucket)")

print()
print("💡 In production you'd identify clients by authenticated user ID, OAuth")
print("   token, or API key — not by raw IP address.")


alice 11th request : 429  (alice exhausted her quota)
bob   1st  request : 200  (bob has his own bucket)

💡 In production you'd identify clients by authenticated user ID, OAuth
   token, or API key — not by raw IP address.


## 📚 Other Rate Limiting Algorithms

Fixed window is the simplest, but there are better algorithms. Here's a comparison:

### 1. Fixed Window (what we used)
- **How it works:** Divide time into fixed windows, count requests in each
- ✅ Simple to implement
- ❌ Burst problem at window boundaries (we demonstrated this above)

### 2. Sliding Window 🪟
- **How it works:** Instead of fixed windows, look at the **last N seconds** from right now
- Example: "Have there been 10 requests in the last 60 seconds?" (not "in this minute")
- ✅ No burst problem — smoothly limits traffic
- ❌ More memory needed (must track individual request timestamps)

### 3. Token Bucket 🪣
- **How it works:** Imagine a bucket that holds tokens. Tokens are added at a fixed rate (e.g., 1 per second). Each request costs 1 token. No tokens left? Request rejected.
- ✅ Allows controlled bursts (if tokens have accumulated)
- ✅ Smooth and flexible
- ❌ Slightly more complex to implement

### 4. Leaky Bucket 🚰
- **How it works:** Requests go into a queue (the bucket). They're processed at a fixed rate (the "leak"). If the bucket is full, new requests are rejected.
- ✅ Very smooth output rate — no bursts at all
- ❌ Doesn't allow any bursting, even if the server could handle it

### Comparison Table

| Algorithm | Complexity | Burst Handling | Memory Usage | Best For |
|-----------|-----------|----------------|-------------|----------|
| **Fixed Window** | ⭐ Simple | ❌ Burst at boundary | Low | Simple APIs, getting started |
| **Sliding Window** | ⭐⭐ Medium | ✅ No bursts | Medium | Most production APIs |
| **Token Bucket** | ⭐⭐ Medium | ✅ Controlled bursts | Low | APIs that allow short bursts |
| **Leaky Bucket** | ⭐⭐ Medium | ✅ No bursts | Medium | APIs needing constant rate |

### Which Should You Use?

- **Learning/simple projects:** Fixed window (what we used in this lab)
- **Production APIs:** Sliding window or token bucket
- **High-traffic APIs:** Token bucket (used by AWS, Stripe, and many others)

## 🎯 Key Takeaways

### What We Learned

1. **Rate limiting protects both the server and its clients** — It prevents abuse, ensures fair access, and keeps the server healthy for everyone.

2. **Always check and respect rate-limit headers** — The three key headers are:
   - `X-RateLimit-Limit` → How many requests you're allowed
   - `X-RateLimit-Remaining` → How many you have left
   - `X-RateLimit-Reset` → When your limit resets

3. **Handle 429 responses gracefully** — Use exponential backoff (wait 1s, 2s, 4s...) instead of hammering the server with retries.

4. **Fixed-window is the simplest algorithm** — But it has a burst problem at window boundaries. Sliding window and token bucket are more fair.

5. **In system design interviews** — Always mention rate limiting when designing public-facing APIs. It shows you think about reliability and abuse prevention.

### 💡 Interview Tip

When asked *"How would you design a rate limiter?"* in an interview, mention:
- **Where** to put it (API gateway, middleware, or application level)
- **Which algorithm** to use (token bucket is the most popular answer)
- **Distributed rate limiting** — How to share counters across multiple servers (Redis is a common choice)
- **What happens when a request is rejected** — Return 429 with helpful headers

### 🔗 Further Reading

- [RFC 6585 — 429 Too Many Requests](https://tools.ietf.org/html/rfc6585)
- [Stripe's rate limiting blog post](https://stripe.com/blog/rate-limiters)
- [System Design — Rate Limiter](https://bytebytego.com/courses/system-design-interview/design-a-rate-limiter)